# Ordered Logistic Regression for Rangeland Management: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset on ordered logistic regression results for adoption predictors of indigenous and modern knowledge in rangeland management practices (Northern Kenya)](https://sen.science/doi/10.71728/senscience.y7m0-f273/) using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library and pandas.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not already present
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and access records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}\n")
print(f"License: {metadata.license}")


## 2. Data Overview
Explore available record sets, their `@id`s, and the fields within each record set. All references to entities use their unique `@id`. This helps in identifying the data structures available for further loading and processing.

In [ ]:
# List all record sets in the dataset with their @id and name
print("Record Sets in the dataset:")
record_sets = []
for recset in dataset.record_sets:
    print(f"  - @id: {recset.id}\n    name: {recset.name}")
    record_sets.append(recset)

print("\nRecord Set Details (fields and their @id):")
for recset in record_sets:
    print(f"\nRecord Set Name: {recset.name}")
    print(f"@id: {recset.id}")
    if hasattr(recset, 'fields') and recset.fields:
        print("Fields:")
        for f in recset.fields:
            print(f"  - Field name: {f.name}, @id: {f.id}, dataType: {getattr(f, 'data_type', 'N/A')}")
    else:
        print("  No fields listed in schema.")

## 3. Data Extraction
Extract data from each record set (identified by its `@id`), loading each set into a pandas DataFrame. All data columns are referenced by their `@id` where appropriate.

In [ ]:
# Extract all record sets and load them into pandas DataFrames, referenced by their @id
record_set_ids = [recset.id for recset in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"No records loaded for record set @id: {record_set_id}: {e}")

# Show columns in each (non-empty) dataframe
for rs_id, df in dataframes.items():
    print(f"\nRecord Set @id: {rs_id}\nColumns (@id): {list(df.columns)}\nSample rows:")
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Example: Filter, normalize, or group on numeric/categorical fields within the dataset. For demonstration, select a record set and a numeric field by their `@id` (you can change these based on the actual record sets and fields in your data).

In [ ]:
# Example: Use a record set and fields by @id for EDA
# ---- SET these variables based on the printout above ----
# For illustration, we'll attempt to select the first available record set and a likely numeric field, if present.
if len(dataframes) == 0:
    print("No record sets with data found in the current package.")
else:
    record_set_id = list(dataframes.keys())[0]  # Choose the first loaded record set
    df = dataframes[record_set_id]

    # Try to guess a numeric field
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        # Try to convert columns to numeric if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field_id = col
                    break
            except Exception:
                pass

    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using record set @id: {record_set_id}")
        print(f"Using numeric field @id: {numeric_field_id}\n")

        # Set a numeric threshold for filtering
        threshold = df[numeric_field_id].quantile(0.75)  # Top 25% values

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df[[numeric_field_id]].head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a non-numeric field if present
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")

## 5. Visualization
Visualize the distribution of a numeric column, or the relationship between the normalized field and a grouping variable.

In [ ]:
# Visualization of numeric field (distribution and group comparison)
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) == 0 or (numeric_field_id is None):
    print("No data available for plotting.")
else:
    # 1. Distribution plot
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # 2. Group plot, if group_field_id was found
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id} (filtered records)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to access, query, and visualize data from a Croissant-structured dataset using `mlcroissant`. By referencing all entities via their `@id`, you ensure reproducibility and precision in data manipulation. Further analyses, such as statistical modeling or hypothesis testing, can follow from these prepared and visualized data structures.

**Notes:**
- The structures (record sets, fields) available in a dataset may vary. Refer to the schema exploration code outputs to locate the appropriate `@id`s for your analysis.
- For a full exploration, expand the EDA and visualization sections using knowledge of the dataset's specific record set and field structure, always referencing by `@id`.